# 02. 객실 변수는 기존 변수에 더해 예측 정보를 주는가?

초기 단일 분할 실험에서 객실 변수의 효과가 분명하지 않아, 같은 5개 fold에서 비교했다.
승객 등급에 객실 변수를 추가하는 경우와 Pclass를 Deck·CabinKnown으로 대체하는 경우를 구분한다.
또한 Age를 연령 구간으로 바꾼 뒤에도 객실 변수가 추가 정보를 주는지 확인한다.


> **기록 상태**: 2026년 7월 개인 실험을 공개용으로 정리한 노트북입니다.
> 표시된 표와 수치는 원본에 저장된 과거 출력이며, 이번 정리본을 처음부터 다시 실행한 결과는 아닙니다.
> 입력 경로와 설명을 정리하고 실행 번호를 초기화했습니다. 원본 데이터로 전체 실행한 뒤 실행 환경과 결과를 확인할 수 있습니다.
> 원본·수정 범위는 [기록 정리 내역](../reports/provenance.md)에 있습니다.


## 0. 이번 노트북의 핵심 원칙

Feature를 비교할 때는 다음을 고정한다.

- 같은 5개 fold
- 같은 평가 지표: accuracy
- 같은 모델: 우선 Logistic Regression
- 같은 결측치 처리와 인코딩 방식

Feature 조합을 결정한 뒤에만 모델 종류를 바꾸어 비교한다.

이렇게 해야 성능 차이가 **feature 때문인지 모델 때문인지** 구분할 수 있다.

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

print("Ready")

Ready


## 2. 데이터 불러오기

In [ ]:
from pathlib import Path
import os

# 로컬: 저장소 루트 또는 notebooks/에서 실행. Kaggle: Titanic 데이터를 연결.
# 다른 위치를 쓰려면 TITANIC_DATA_DIR에 train.csv/test.csv가 있는 폴더를 지정한다.
configured_dir = os.environ.get("TITANIC_DATA_DIR")
if configured_dir:
    candidates = [Path(configured_dir)]
else:
    candidates = [
        Path("data"), Path("../data"),
        Path("/kaggle/input/competitions/titanic"),
        Path("/kaggle/input/titanic"),
    ]
data_dir = next(
    (p for p in candidates if (p / "train.csv").is_file() and (p / "test.csv").is_file()),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Titanic train.csv와 test.csv를 data/에 넣거나 TITANIC_DATA_DIR을 지정하세요."
    )
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
assert "Survived" in train.columns and "Survived" not in test.columns
assert train["PassengerId"].is_unique and test["PassengerId"].is_unique
print("train:", train.shape, "test:", test.shape)


## 3. Train과 test에 동일한 feature 만들기

최종 제출 단계에서는 train과 test에 **완전히 같은 규칙**으로 feature를 만들어야 한다.

이 함수는 다음 변수를 생성한다.

- `FamilySize = SibSp + Parch + 1`
- `AgeGroup`
- `CabinKnown`
- `Deck`

`Age`의 결측치는 여기서 직접 채우지 않는다.  
교차검증 중 각 training fold에서 계산한 중앙값으로 채우기 위해 `SimpleImputer`를 Pipeline 안에 둔다.

In [ ]:
def create_features(data):
    result = data.copy()

    # 본인을 포함한 전체 가족 규모
    result["FamilySize"] = (
        result["SibSp"] + result["Parch"] + 1
    )

    # 나이 구간: Age 원값과 비교하기 위한 범주형 파생변수
    result["AgeGroup"] = pd.cut(
        result["Age"],
        bins=[0, 6, 20, 40, 60, np.inf],
        labels=[
            "Preschool",
            "Teen",
            "20-39",
            "40-59",
            "60+"
        ],
        right=False # pd.cut()은 숫자를 여러 구간(bin)으로 나눔. 기본(right=True)으로 (a, b] 이런 형태. False는 [a,b) 이런 형태
    )

    result["AgeGroup"] = (
        result["AgeGroup"]
        .astype("object") #"AgeGroup" 이 category 형식인데, "Unknown"이라는 새로운 category를 넣고 싶어서 기존 category(Teen, 20-39)들을 문자열("object")로 바꾸고 "Unknown" 추가
        .fillna("Unknown")
    )

    # Cabin 정보가 있으면 1, 없으면 0
    result["CabinKnown"] = (
        result["Cabin"].notna().astype(int)
    )

    # Cabin 문자열 맨 앞의 알파벳을 Deck으로 사용
    result["Deck"] = (
        result["Cabin"]
        .str.extract(r"^([A-Za-z])", expand=False) # r"^([A-Za-z])": 문자열(^)의 첫 번째 글자가 알파벳([A-Za-z])인가?/ expand=False: 추출 결과를 DataFrame이 아니라 Series로 반환
        .fillna("Unknown")
    )

    return result


train_fe = create_features(train)
test_fe = create_features(test)

# train_fe[
#     [
#         "Sex",
#         "Pclass",
#         "Age",
#         "AgeGroup",
#         "FamilySize",
#         "CabinKnown",
#         "Deck",
#         "Survived"
#     ]
# ].head()

## 4. 전처리와 모델 Pipeline

수치형 결측은 학습 fold의 중앙값으로 채우고 표준화한다.
범주형 결측은 학습 fold의 최빈값으로 채운 뒤 one-hot 인코딩한다.
다만 이 노트북의 `AgeGroup` 결측은 feature 생성 단계에서 `Unknown`으로 이미 분리된다.
`Deck` 결측도 `Unknown`이다. 이 선택이 가장 좋은지는 별도로 비교하지 않았다.

학습이 필요한 전처리를 Pipeline 안에 두어 검증 fold의 정보를 사용하지 않게 한다.


In [ ]:
def build_preprocessor(
    numeric_features,
    categorical_features
):
    transformers = []

    if numeric_features:
        numeric_pipeline = Pipeline(steps=[
            (
                "imputer",
                SimpleImputer(strategy="median") 
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

        transformers.append(
            (
                "num",
                numeric_pipeline,
                numeric_features
            )
        )

    if categorical_features:
        categorical_pipeline = Pipeline(steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ])

        transformers.append(
            (
                "cat",
                categorical_pipeline,
                categorical_features
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


#이걸로 전처리된 데이터를 -> estimator 모델에 대입
def build_model_pipeline( 
    numeric_features,
    categorical_features,
    estimator
):
    preprocessor = build_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features
    )

    return Pipeline(steps=[
        (
            "preprocessor", preprocessor  #전처리하고 
        ),
        (
            "classifier", estimator  #모델 돌려라
        )
    ])

## 5. 5-fold Stratified Cross-Validation 설정

`StratifiedKFold`는 각 fold의 생존자·사망자 비율을 원자료와 비슷하게 유지한다.

- 전체 데이터를 5개 fold로 나눈다. (80/20)
- 네 fold로 학습하고 한 fold로 검증한다.
- 검증 fold를 바꾸어 총 5번 반복한다.
- 다섯 accuracy의 평균과 표준편차를 확인한다.

In [ ]:
X_all = train_fe.drop(columns="Survived")
y = train_fe["Survived"]

cv = StratifiedKFold( # 각 fold마다 추후에 y를 받으면 그 비율이 비슷하게 division
    n_splits=5, # fold의 개수 
    shuffle=True, # 처음 데이터의 정렬 순서를 shuffle해서 fold를 나눔
    random_state=42 # convention으로 42를 사용 
)

### 한 feature 실험을 5-fold로 평가하는 함수

In [ ]:
# sklearn-cross_val_score()

Baseline=["Sex","Pclass", "FamilySize", "Age"]

X=train_fe[Baseline]
#y=train_fe["Survived"]


#cv = StratifiedKFold(
#    n_splits=5,
#    shuffle=True,
#    random_state=42
#)


Baseline_pipeline = build_model_pipeline(
    numeric_features=["FamilySize", "Age"],
    categorical_features=["Sex", "Pclass"],
    estimator=LogisticRegression(max_iter=1000)
)

manual_scores = []

for fold_num, (train_idx, valid_idx) in enumerate(cv.split(X,y), start=1): #cv.split(X,y): 여기서 StratifiedKFold에서 y에 대해 비슷한 비율로 5 partition/ #enumerate: 반복 횟수(fold 번호)와 (train_idx, valid_idx)를 함께 꺼냄  
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    fold_pipeline = clone(Baseline_pipeline)

    fold_pipeline.fit(X_train, y_train)
    valid_pred = fold_pipeline.predict(X_valid)

    fold_accuracy = accuracy_score(
        y_valid, valid_pred
    )

    manual_scores.append(fold_accuracy)

manual_scores

sum(manual_scores)/len(manual_scores)

0.7923545289059067

In [ ]:
def evaluate_with_cv(
    experiment_name,
    features,
    numeric_features,
    categorical_features,
    estimator
):
    X = train_fe[features]

    pipeline = build_model_pipeline(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        estimator=clone(estimator) # clone ; 아직 학습되지 않은 새 모델을 만든다. (기존의 학습된 상태나 내부 값이 실험 간에 섞이지 않도록 하는 안전장치) 
    )

    # sklearn-cross_val_score function
    scores = cross_val_score(
        estimator=pipeline,
        X=X,
        y=y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1 # 사용 가능한 CPU 코어를 모두 사용해 교차검증을 병렬 실행 (n_jobs=1이면 한 번에 하나씩 실행)
    )

    result = {
        "experiment": experiment_name,
        "features": ", ".join(features),
        "fold_1": scores[0],
        "fold_2": scores[1],
        "fold_3": scores[2],
        "fold_4": scores[3],
        "fold_5": scores[4],
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    }

    return result

scores_comparison= evaluate_with_cv(
    'Trial',
    features=Baseline,
    numeric_features=["FamilySize", "Age"],
    categorical_features=["Sex", "Pclass"],
    estimator=LogisticRegression(max_iter=1000)
)

scores_comparison

{'experiment': 'Trial',
 'features': 'Sex, Pclass, FamilySize, Age',
 'fold_1': np.float64(0.8044692737430168),
 'fold_2': np.float64(0.7921348314606742),
 'fold_3': np.float64(0.7865168539325843),
 'fold_4': np.float64(0.7752808988764045),
 'fold_5': np.float64(0.8033707865168539),
 'mean_accuracy': np.float64(0.7923545289059067),
 'std_accuracy': np.float64(0.010897344012785441)}

## 6. 같은 Cabin feature/Age 실험을 5-fold로 반복

이제 Logistic Regression을 고정하고 feature 조합만 바꾼다.

비교할 조합:

1. Baseline
2. Baseline + CabinKnown
3. Baseline + Deck
4. Baseline + CabinKnown + Deck
5. Age를 AgeGroup으로 교체
6. AgeGroup + CabinKnown + Deck
7. Pclass를 제거하고 Deck + CabinKnown으로 대체

마지막 실험은 EDA에서 관찰한 `Pclass`와 `Deck/CabinKnown`의 정보 중복을 확인하기 위한 것이다.

In [ ]:
# AgeGroup 실험용 목록을 만든다.
base_numeric=["FamilySize", "Age"]
base_categorical=["Sex", "Pclass"]

agegroup_numeric = [
    feature
    for feature in base_numeric
    if feature != "Age"
]

agegroup_categorical = (
    base_categorical + ["AgeGroup"]
)

feature_experiments = [
    {
        "name": "Baseline",
        "features": Baseline,
        "numeric": base_numeric,
        "categorical": base_categorical
    },
    {
        "name": "Baseline + CabinKnown",
        "features": Baseline + ["CabinKnown"],
        "numeric": base_numeric + ["CabinKnown"],
        "categorical": base_categorical
    },
    {
        "name": "Baseline + Deck",
        "features": Baseline + ["Deck"],
        "numeric": base_numeric,
        "categorical": base_categorical + ["Deck"]
    },
    {
        "name": "Baseline + CabinKnown + Deck",
        "features": Baseline + ["CabinKnown", "Deck"],
        "numeric": base_numeric + ["CabinKnown"],
        "categorical": base_categorical + ["Deck"]
    },
    {
        "name": "Baseline with AgeGroup",
        "features": [
            "Sex",
            "Pclass",
            "FamilySize",
            "AgeGroup"
        ],
        "numeric": agegroup_numeric,
        "categorical": agegroup_categorical
    },
    {
        "name": "AgeGroup + CabinKnown + Deck",
        "features": [
            "Sex",
            "Pclass",
            "FamilySize",
            "AgeGroup",
            "CabinKnown",
            "Deck"
        ],
        "numeric": agegroup_numeric + ["CabinKnown"],
        "categorical": agegroup_categorical + ["Deck"]
    },
    {
        "name": "Deck + CabinKnown replace Pclass",
        "features": [
            "Sex",
            "FamilySize",
            "Age",
            "CabinKnown",
            "Deck"
        ],
        "numeric": [
            "FamilySize",
            "Age",
            "CabinKnown"
        ],
        "categorical": [
            "Sex",
            "Deck"
        ]
    }
]

In [ ]:
feature_cv_results = []

for experiment in feature_experiments:
    result = evaluate_with_cv(
        experiment_name=experiment["name"],
        features=experiment["features"],
        numeric_features=experiment["numeric"],
        categorical_features=experiment["categorical"],
        estimator=LogisticRegression(max_iter=1000)
    )

    feature_cv_results.append(result)


# feature_cv_results # dictionary type
feature_cv_table = (
    pd.DataFrame(feature_cv_results)
    .sort_values("mean_accuracy", ascending=False)
    .reset_index(drop=True)
)

feature_cv_table

,experiment,features,fold_1,fold_2,fold_3,fold_4,fold_5,mean_accuracy,std_accuracy
0,Baseline with AgeGroup,"Sex, Pclass, FamilySize, AgeGroup",0.815642,0.808989,0.808989,0.803371,0.853933,0.818185,0.018292
1,AgeGroup + CabinKnown + Deck,"Sex, Pclass, FamilySize, AgeGroup, CabinKnown,...",0.815642,0.825843,0.780899,0.803371,0.837079,0.812567,0.019360
2,Baseline + CabinKnown,"Sex, Pclass, FamilySize, Age, CabinKnown",0.821229,0.814607,0.769663,0.780899,0.837079,0.804695,0.025353
3,Baseline + Deck,"Sex, Pclass, FamilySize, Age, Deck",0.810056,0.820225,0.764045,0.775281,0.837079,0.801337,0.027495
4,Baseline + CabinKnown + Deck,"Sex, Pclass, FamilySize, Age, CabinKnown, Deck",0.810056,0.814607,0.758427,0.780899,0.837079,0.800213,0.027504
5,Deck + CabinKnown replace Pclass,"Sex, FamilySize, Age, CabinKnown, Deck",0.810056,0.803371,0.764045,0.786517,0.814607,0.795719,0.018487
6,Baseline,"Sex, Pclass, FamilySize, Age",0.804469,0.792135,0.786517,0.775281,0.803371,0.792355,0.010897


### 결과를 읽는 기준

평균 정확도와 fold별 변동을 함께 본다. 표준편차는 각 fold 점수의 산포이며,
평균에서 빼는 벌점이나 통계적 유의성 검정 결과가 아니다.
아래 상위 두 실험의 차이만으로 어느 방식이 일반적으로 우월하다고 단정하지 않는다.


In [ ]:
top_two = feature_cv_table.head(2).copy()

if len(top_two) == 2:
    mean_gap = (
        top_two.loc[0, "mean_accuracy"]
        - top_two.loc[1, "mean_accuracy"]
    )

    print("상위 두 실험 평균 accuracy 차이:", round(mean_gap, 6))

top_two[
    [
        "experiment",
        "mean_accuracy",
        "std_accuracy"
    ]
]

상위 두 실험 평균 accuracy 차이: 0.005618


,experiment,mean_accuracy,std_accuracy
0,Baseline with AgeGroup,0.818185,0.018292
1,AgeGroup + CabinKnown + Deck,0.812567,0.019360


## 7. 관찰한 결과와 다음 단계

- Age 원값을 쓴 기준 모델의 평균 정확도는 0.792355였다. CabinKnown을 더하면 0.804695였다.
- AgeGroup을 쓴 기준 모델에서는 0.818185였고, CabinKnown·Deck을 함께 더하면 0.812567이었다.
- Pclass를 Deck·CabinKnown으로 대체한 실험은 Age 원값을 쓰며, 평균 정확도는 0.795719였다.

따라서 "객실 정보는 쓸모없다"가 아니라 **AgeGroup을 포함한 이번 기준 모델에서는
객실 변수 두 개를 함께 추가해도 평균 정확도가 높아지지 않았다**고 정리했다.
EDA의 생존율 차이와 다른 변수에 더해 주는 예측 정보는 서로 다른 질문이었다.

다음 실험에서는 Sex·Pclass·FamilySize·AgeGroup을 유지하고 분류 기준과 class weight를 비교했다.
단, 다음 노트북은 AgeGroup 결측을 학습 fold의 최빈값으로 처리하므로
기준 정확도 차이를 threshold의 효과로 해석하면 안 된다.
이는 당시 실험 조건의 차이이며, 두 결측 처리법을 통제해 비교한 실험은 아니다.

같은 CV 결과를 보며 변수를 선택했으므로 이 점수는 독립적인 최종 평가 점수가 아니다.

다음: [03. Threshold와 class weight 비교](03_threshold_class_weight.ipynb)


## 실행 환경 기록

전체 재실행 시 아래 버전을 결과와 함께 보관합니다.


In [ ]:
import platform
from importlib.metadata import version

print("Python", platform.python_version())
for package in ["numpy", "pandas", "scikit-learn", "matplotlib"]:
    print(package, version(package))
